# Define

missing value ko fill karta hai nearest neighbors ke values ka average lekar.

# When To Use KNN Imputer

Use when:

Data is not too large
Rows are similar to each other
Better than mean/median because it uses pattern, not just average

# Why KNN Imputer is better than mean imputation?

Because KNN uses similar data points to fill missing values, so it gives more accurate and realistic values compared to mean.

# Code


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('train.csv')[['Age','Pclass','Fare','Survived']]

In [3]:
df.head()

,Age,Pclass,Fare,Survived
0,22.0,3,7.2500,0
1,38.0,1,71.2833,1
2,26.0,3,7.9250,1
3,35.0,1,53.1000,1
4,35.0,3,8.0500,0


In [4]:
df.isnull().mean() * 100

,0
Age,19.86532
Pclass,0.00000
Fare,0.00000
Survived,0.00000


In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [7]:
X_train.head()

,Age,Pclass,Fare
30,40.0,1,27.7208
10,4.0,3,16.7000
873,47.0,3,9.0000
182,9.0,3,31.3875
876,20.0,3,9.8458


In [12]:
knn = KNNImputer(n_neighbors=3,weights='distance') # these line code means : Fill missing values using the 3 nearest rows, and give more importance to closer rows.

# weights = 'distance' :

# 1 Closer neighbors have more influence
# 2 Far neighbors have less influence

# Weight  = 1/Distance

# Distance small → Weight big → More important

# Distance large → Weight small → Less important

X_train_trf = knn.fit_transform(X_train)
X_test_trf = knn.transform(X_test)


In [9]:

lr = LogisticRegression()

lr.fit(X_train_trf,y_train)

y_pred = lr.predict(X_test_trf)

accuracy_score(y_test,y_pred)

0.7039106145251397

In [10]:
# Comparision with Simple Imputer --> mean

si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [11]:
lr = LogisticRegression()

lr.fit(X_train_trf2,y_train)

y_pred2 = lr.predict(X_test_trf2)

accuracy_score(y_test,y_pred2)

0.6927374301675978

# Important Point

# “We use Logistic Regression because the target variable is categorical (0/1). Linear Regression is used when the target variable is continuous. Imputation method is independent of the model.”

# Imputer choice does NOT depend on Logistic or Linear Regression.

# It depends on missing data, not on model.


# Step 1 — Check Target Variable (y)

Target (y)	Problem Type	Model

Continuous (price, salary, marks) --	Regression	Linear Regression

Categorical (0/1, Yes/No, Pass/Fail) --	Classification	Logistic Regression

So:

If y = continuous → Linear Regression

If y = 0/1 → Logistic Regression

Model depends on TARGET, not on imputer.

# Step 2 — Where Imputer is Used?

Imputer is used for X (input features), not for y.

Pipeline:

# X (missing values) → Imputer → Clean X → Model → Predict y

# Why scaling is important in KNN?

Because KNN uses distance. If features are on different scales, large scale features will dominate distance calculation.

Example:

Age: 20–50

Salary: 20,000–80,000 → Salary dominates → wrong neighbors

So we use:

StandardScaler
MinMaxScaler

# **(KNN Imputer fills missing values using the values of the nearest neighbors. It does not simply take the overall average like mean imputation. Instead, it finds similar data points and uses their values, so it is often more accurate. When we use weights='distance', closer neighbors have more influence than farther neighbors.)**

# ITERATIVE IMPUTER

Iterative Imputer is a smart missing value filling method.

Simple words:

It treats missing value like a prediction problem and uses ML model to predict it.

# MICE  = Multivariable Imputer chained Equations

Intuition Example

Suppose data:

Age	Experience	Salary
25	2	30k
26	3	NaN
24	2	32k
40	15	80k

Salary depends on Age + Experience.

So Iterative Imputer will:

Build a model:
Salary = f(Age, Experience)
Predict missing salary → Fill NaN

So instead of average or neighbors → it uses regression model.



# How Iterative Imputer Works (Algorithm)

First fill missing values with mean (initial guess)

Pick one column with missing values (e.g., Salary)

Treat that column as target

Use other columns as features

Train regression model

Predict missing values

Move to next column

Repeat multiple times (iterations)

Final filled dataset returned

# Difference: Mean vs KNN vs Iterative
Method	   How it fills

Mean	     Column average

KNN	       Nearest neighbors

Iterative	 ML model prediction

# Iterative is slow
# But very accurate
 # Why?

 Because Iterative Imputer does prediction using ML model, not average.

 # Why It Is Slow

 Because it trains a model again and again.



# **(Iterative Imputer is slow because it repeatedly trains machine learning models for each feature with missing values, but it is accurate because it uses relationships between features to predict missing values.)**



# Code



In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

In [13]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [33]:
df = df.iloc[:,0:-1].copy()
df

,R&D Spend,Administration
21,8.0,15.0
37,NaN,5.0
2,15.0,10.0
14,12.0,NaN
44,2.0,15.0


In [34]:
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan

In [30]:
df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


In [36]:
# Step 1 - Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())

In [37]:
# 0th Iteration
df0

,R&D Spend,Administration
21,8.00,15.0
37,9.25,5.0
2,15.00,10.0
14,12.00,10.0
44,2.00,10.0


In [41]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.nan

df1

,R&D Spend,Administration
21,8.0,15.0
37,NaN,5.0
2,15.0,10.0
14,12.0,10.0
44,2.0,10.0


In [42]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration
21,15.0
2,10.0
14,10.0
44,10.0


In [43]:
y = df1.iloc[[0,2,3,4],0]
y

,R&D Spend
21,8.0
2,15.0
14,12.0
44,2.0


In [49]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

NameError: name 'LinearRegression' is not defined

In [46]:
df1.iloc[1,0] = 23.14

In [47]:
df1

,R&D Spend,Administration
21,8.00,15.0
37,23.14,5.0
2,15.00,10.0
14,12.00,10.0
44,2.00,10.0


In [50]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.nan

df1

,R&D Spend,Administration
21,8.00,15.0
37,23.14,5.0
2,15.00,10.0
14,12.00,NaN
44,2.00,10.0


In [52]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]]
X

IndexError: positional indexers are out-of-bounds

In [53]:
y =  df1.iloc[[0,1,2,4],1]
y

,Administration
21,15.0
37,5.0
2,10.0
44,10.0


In [54]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

NameError: name 'LinearRegression' is not defined

In [55]:
df1.iloc[3,1] = 11.06

In [56]:
df1

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06
44,2.00,10.00


In [57]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.NaN

df1

AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [58]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06


In [59]:
y = df1.iloc[0:4,-1]
y

,Administration
21,15.00
37,5.00
2,10.00
14,11.06


In [60]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

NameError: name 'LinearRegression' is not defined

In [61]:
df1.iloc[4,-1] = 31.56

In [62]:
# After 1st Iteration
df1

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06
44,2.00,31.56


In [63]:
 # Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration
21,0.00,0.00
37,13.89,0.00
2,0.00,0.00
14,0.00,1.06
44,0.00,21.56


In [64]:
df2 = df1.copy()

df2.iloc[1,0] = np.NaN

df2

AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [65]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

NameError: name 'LinearRegression' is not defined

In [66]:
df2.iloc[1,0] = 23.78

In [67]:
df2.iloc[3,1] = np.NaN
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [68]:
df2.iloc[3,1] = 11.22

In [69]:

df2.iloc[4,-1] = np.NaN

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [70]:
df2.iloc[4,-1] = 31.56

In [71]:
df2

,R&D Spend,Administration
21,8.00,15.00
37,23.78,5.00
2,15.00,10.00
14,12.00,11.22
44,2.00,31.56


In [72]:

df2 - df1

,R&D Spend,Administration
21,0.00,0.00
37,0.64,0.00
2,0.00,0.00
14,0.00,0.16
44,0.00,0.00


In [73]:
df3 = df2.copy()

df3.iloc[1,0] = np.NaN

df3

AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [74]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

NameError: name 'LinearRegression' is not defined

In [75]:
df3.iloc[1,0] = 24.57

In [76]:

df3.iloc[4,-1] = np.NaN

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [77]:
df3.iloc[4,-1] = 45.53

In [78]:
df2.iloc[3,1] = 11.22

In [79]:
df3

,R&D Spend,Administration
21,8.00,15.00
37,24.57,5.00
2,15.00,10.00
14,12.00,11.22
44,2.00,45.53


In [80]:

df3 - df2

,R&D Spend,Administration
21,0.00,0.00
37,0.79,0.00
2,0.00,0.00
14,0.00,0.00
44,0.00,13.97
